In [196]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [197]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Experimental conditions

In [225]:
silver_path = Path.cwd().parent / 'data/silver_layer/'

silver_confocal_path = Path.cwd().parent / 'data/silver_layer/confocal_runs'
gold_profile_path = Path.cwd().parent / 'data/gold_layer/profile_pics'

profile_01_path = gold_profile_path / 'profile_pics_01'
profile_02_path = gold_profile_path / 'profile_pics_02'
profile_full_path = gold_profile_path / 'profile_pics_full'

gold_profile_path.mkdir(parents=True, exist_ok=True)
profile_01_path.mkdir(parents=True, exist_ok=True)
profile_02_path.mkdir(parents=True, exist_ok=True)
profile_full_path.mkdir(parents=True, exist_ok=True)


In [199]:
df_confocal = pd.read_pickle(silver_path / 'confocal_results.pkl')
df_experimental = pd.read_pickle(silver_path / 'experimental_results.pkl')

# Selected conditions

In [200]:
missing_threshold = 50

In [201]:
list_runs = df_confocal.loc[df_confocal['t2_missing_percentage']<missing_threshold, 'run_id'].unique()

In [202]:
list_runs = df_confocal['run_id'].unique()

In [203]:
df_selected = df_experimental.loc[df_experimental['run_id'].isin(list_runs)]

In [ ]:
plt.rcParams.update({

    # -------------------------------------------------
    # LaTeX
    # -------------------------------------------------

    "text.usetex": True,

    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],

    # -------------------------------------------------
    # Font sizes
    # -------------------------------------------------

    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,

    # -------------------------------------------------
    # Axes
    # -------------------------------------------------

    "axes.linewidth": 0.6,

    # -------------------------------------------------
    # Ticks
    # -------------------------------------------------

    "xtick.direction": "in",
    "ytick.direction": "in",

    "xtick.major.size": 3,
    "ytick.major.size": 3,

    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,

    # -------------------------------------------------
    # EPS / PS backend
    # -------------------------------------------------

    "ps.usedistiller": "xpdf",

    # Important:
    # avoids Type 3 fonts
    "ps.fonttype": 42,

    # Also useful sometimes
    "pdf.fonttype": 42,

    # -------------------------------------------------
    # Savefig
    # -------------------------------------------------

    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
})

# =========================================================
# Figure dimensions
# =========================================================

CM = 1 / 2.54

FIG_WIDTH_CM = 8.4
FIG_HEIGHT_CM = 3.2 # 4.2

FIG_SIZE = (
    FIG_WIDTH_CM * CM,
    FIG_HEIGHT_CM * CM
)

PLOT_CONFIG = {

    # Geometry
    "glass_thickness": 0.96,
    "channel_height": 8.02,

    # Axis limits
    "ymin": -1.2,
    "ymax": 8.3,

    # Optional crop
    "min_time": 0,
    "max_time": None,

    # Corrections
    "t3_threshold": 1.3,
    "t3_scale_num": 1.51,
    "t3_scale_den": 1.33,

    # Plot style
    "wall_color": "black",
    "interface_color": "black",

    "wall_linewidth": 0.7,
    "interface_linewidth": 0.7,

    "interface_alpha": 1.0,

    "interface_color": "#1f77b4",

    "use_markers": False,
    "marker_size": 1.0,
    "marker_every": 1,

}

PLOT_CONFIG_01 = {
    **PLOT_CONFIG,
    "max_time": 1,
}

PLOT_CONFIG_02 = {
    **PLOT_CONFIG,
    "max_time": 2,
}

In [205]:
# =========================================================
# Utilities
# =========================================================

def cm_to_inch(value_cm):
    return value_cm / 2.54


def load_confocal_run(run_id, base_path=silver_confocal_path):
    """
    Load confocal run CSV.

    Expected columns:
        time, t1, t2, t3
    """

    file_path = base_path / f'confocal_run_{run_id}.csv'

    df = pd.read_csv(file_path)

    return df

In [206]:
def preprocess_profile(df, config=PLOT_CONFIG):
    """
    Apply legacy corrections and preprocessing.
    """

    df = df.copy()

    # -----------------------------------------------------
    # Remove invalid measurements
    # -----------------------------------------------------
    df = df.loc[~df['t1'].isna()].copy()

    # -----------------------------------------------------
    # Normalize time
    # -----------------------------------------------------
    df['time'] = df['time'] - df['time'].min()

    # -----------------------------------------------------
    # Fill missing t3
    # -----------------------------------------------------
    df.loc[df['t3'].isna(), 't3'] = config['glass_thickness']

    # -----------------------------------------------------
    # Legacy t3 correction
    # -----------------------------------------------------
    mask_t3 = df['t3'] > config['t3_threshold']

    df.loc[mask_t3, 't3'] = (
        df.loc[mask_t3, 't3']
        * config['t3_scale_num']
        / config['t3_scale_den']
    )

    # -----------------------------------------------------
    # Overflow correction
    # -----------------------------------------------------
    mask_overflow = df['t2'] > config['channel_height']

    df.loc[mask_overflow, 't3'] = (
        (df.loc[mask_overflow, 't2'] - config['channel_height'])
        * config['t3_scale_num']
        / config['t3_scale_den']
    )

    df.loc[mask_overflow, 't2'] = config['channel_height']

    # -----------------------------------------------------
    # Bottom wall
    # -----------------------------------------------------
    df['t0'] = -config['glass_thickness']

    # -----------------------------------------------------
    # Optional time crop
    # -----------------------------------------------------
    if config['min_time'] is not None:
        df = df.loc[df['time'] >= config['min_time']]

    if config['max_time'] is not None:
        df = df.loc[df['time'] <= config['max_time']]

    return df

In [207]:
def create_profile_plot(
    df,
    run_id,
    config=PLOT_CONFIG,
    show_xlabel=True,
    show_title=False,
):

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    # -----------------------------------------------------
    # Walls
    # -----------------------------------------------------

    ax.plot(
        df['time'].values,
        df['t0'].values,
        color=config['wall_color'],
        linewidth=config['wall_linewidth']
    )

    ax.plot(
        df['time'].values,
        (df['t1'] - config['glass_thickness']).values,
        color=config['wall_color'],
        linewidth=config['wall_linewidth']
    )

    ax.plot(
        df['time'].values,
        np.full(len(df), config['channel_height'] + 0.1),
        color=config['wall_color'],
        linewidth=config['wall_linewidth']
    )

    # -----------------------------------------------------
    # Interface
    # -----------------------------------------------------

    plot_kwargs = {
        "color": config['interface_color'],
        "linewidth": config['interface_linewidth'],
        # "alpha": config['interface_alpha'],
    }

    if config['use_markers']:

        plot_kwargs.update({
            "marker": "o",
            "markersize": config['marker_size'],
            "markevery": config['marker_every'],
            "markeredgewidth": 0,
        })

    ax.plot(
        df['time'].values,
        df['t2'].values,
        **plot_kwargs
    )

    # -----------------------------------------------------
    # Labels
    # -----------------------------------------------------

    if show_xlabel:
        ax.set_xlabel(r'Time [s]')

    ax.set_ylabel(r'Profile [mm]')

    # -----------------------------------------------------
    # Limits
    # -----------------------------------------------------

    ax.set_ylim(
        config['ymin'],
        config['ymax']
    )

    # -----------------------------------------------------
    # Ticks
    # -----------------------------------------------------

    ax.yaxis.set_major_locator(
        MaxNLocator(nbins=4)
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(nbins=5)
    )

    # -----------------------------------------------------
    # Style
    # -----------------------------------------------------

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # -----------------------------------------------------
    # Optional title
    # -----------------------------------------------------

    if show_title:
        ax.set_title(f'Run {run_id}')

    return fig

In [208]:
def save_profile_plot(
    fig,
    run_id,
    output_path=gold_profile_path,
    extension='eps',
):

    save_path = (
        output_path
        / f'profile_run_{run_id}.{extension}'
    )

    fig.savefig(
        save_path,
        format='eps'
    )

    plt.close(fig)

    return save_path

In [214]:
# =========================================================
# Batch processing
# =========================================================

def process_profile_runs(
    run_list,
    config=PLOT_CONFIG,
    input_path=silver_confocal_path,
    output_path=gold_profile_path,
):
    """
    Process multiple confocal runs.
    """

    for run_id in run_list:

        print(f'Processing run {run_id}...')

        # ---------------------------------------------
        # Load
        # ---------------------------------------------
        df = load_confocal_run(
            run_id=run_id,
            base_path=input_path
        )

        # ---------------------------------------------
        # Preprocess
        # ---------------------------------------------
        df = preprocess_profile(
            df=df,
            config=config
        )

        # ---------------------------------------------
        # Plot
        # ---------------------------------------------
        fig = create_profile_plot(
            df=df,
            run_id=run_id,
            config=config
        )

        # ---------------------------------------------
        # Save
        # ---------------------------------------------
        save_profile_plot(
            fig=fig,
            run_id=run_id,
            output_path=output_path
        )

        print(f'Finished run {run_id}')


In [210]:
# teste = load_confocal_run(
#             run_id=13,
#             base_path=silver_confocal_path
#         )

# teste2 = preprocess_profile(teste, PLOT_CONFIG)

# fig = create_profile_plot(
#     df=teste2,
#     run_id=13,
#     config=PLOT_CONFIG
# )

# save_profile_plot(
#     fig=fig,
#     run_id=13,
#     output_path=gold_profile_path
# )

In [211]:
process_profile_runs(
    run_list=list_runs,
    config=PLOT_CONFIG_01,
    input_path=silver_confocal_path,
    output_path=profile_01_path,
    )

In [212]:
process_profile_runs(
    run_list=list_runs,
    config=PLOT_CONFIG_02,
    input_path=silver_confocal_path,
    output_path=profile_02_path,
    )

In [215]:
process_profile_runs(
    run_list=list_runs,
    config=PLOT_CONFIG,
    input_path=silver_confocal_path,
    output_path=profile_full_path,
    )

Processing run 1...
Finished run 1
Processing run 2...
Finished run 2
Processing run 3...
Finished run 3
Processing run 4...
Finished run 4
Processing run 5...
Finished run 5
Processing run 6...
Finished run 6
Processing run 7...
Finished run 7
Processing run 8...
Finished run 8
Processing run 9...
Finished run 9
Processing run 10...
Finished run 10
Processing run 11...
Finished run 11
Processing run 12...
Finished run 12
Processing run 13...
Finished run 13
Processing run 14...
Finished run 14
Processing run 15...
Finished run 15
Processing run 16...
Finished run 16
Processing run 17...
Finished run 17
Processing run 18...
Finished run 18
Processing run 19...
Finished run 19
Processing run 20...
Finished run 20
Processing run 21...
Finished run 21
Processing run 22...
Finished run 22
Processing run 23...
Finished run 23
Processing run 24...
Finished run 24
Processing run 25...
Finished run 25
Processing run 26...
Finished run 26
Processing run 27...
Finished run 27
Processing run 28..

In [ ]:
# teste = load_confocal_run(
#             run_id=13,
#             base_path=silver_confocal_path
#         )

# teste2 = preprocess_profile(teste, PLOT_CONFIG)

# fig = create_profile_plot(
#     df=teste2,
#     run_id=13,
#     config=PLOT_CONFIG
# )

# save_profile_plot(
#     fig=fig,
#     run_id=13,
#     output_path=gold_profile_path
# )

In [ ]:
# Get-ChildItem *.eps | ForEach-Object {
#     epstopdf $_.FullName
# }